In [16]:
import numpy as np
import pandas as pd
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.plots import plot_convergence

In [45]:
#edit the P array
def edit_P_with_no_Camber(P, P_noCamber):
    P[0] = P_noCamber[0]
    P[1] = P_noCamber[1]
    P[2] = P_noCamber[2]
    P[3] = P_noCamber[3]
    P[5] = P_noCamber[4]
    P[6] = P_noCamber[5]
    P[7] = P_noCamber[6]
    P[8] = P_noCamber[7]
    P[9] = P_noCamber[8]
    P[10] = P_noCamber[9]
    P[12] = P_noCamber[10]
    P[15] = P_noCamber[11]
    return P

def edit_P_with_camber(P, P_camber):
    P[4] = P_camber[0]
    P[11] = P_camber[1]
    P[13] = P_camber[2]
    P[14] = P_camber[3]
    P[16] = P_camber[4]
    P[17] = P_camber[5]
    P[18] = P_camber[6]
    return P

In [57]:
def pacejka(P):
    #P = [250, 1.4, 2.4, -0.25, 3, -0.1, -1.5, 0, 0, -30.5, 1.15, 1, 0, 0, -0.128, 0, 0, 0, 1.43];
    #L = [1, 1, 1, 1, 1, 1, 1, 1];
    dfz = (FZ - P[0]) / P[0]
    Svy = FZ * (P[15] + P[16] * dfz + (P[17] + P[18] * dfz) * IA)
    Ey = (P[5] + P[6] * dfz) * (1 - (P[7] + P[8] * IA) * np.sign(alpha))
    Shy = (P[12] + P[13] * dfz + P[14] * IA)
    alphaY = alpha + Shy
    Cy = P[1]
    Dy = FZ * (P[2] + P[3] * dfz) * (1 - P[4] * IA**2)
    x1 = 2 * np.arctan(FZ / (P[10] * P[0]))
    x2 = P[9] * P[0] * np.sin(x1) * (1 - P[11] * np.abs(IA))
    By = x2 / (Cy * Dy)

    x3 = By * alphaY
    FY = Dy * np.sin(Cy * np.arctan(x3 - Ey * (x3 - np.arctan(x3)))) + Svy

    return FY

def diffPacejka_noCamber(P_noCamber):
    P_camber = np.array([
        P[4],
        P[11],
        P[13],
        P[14],
        P[16],
        P[17],
        P[18]
    ])
    FY_calculated = pacejka(P_noCamber, P_camber, L, FZ, IA, alpha)
    FY_diff_total = 0
    for i in range(len(FY_measured)):
        FY_diff_total += abs(FY_calculated[i] - FY_measured[i])
    print(FY_diff_total / len(FY_measured))
    return FY_diff_total / len(FY_measured)
    
def diffPacejka_camber(P_camber):
    P_noCamber = np.array([
        P[0],
        P[1],
        P[2],
        P[3],
        P[5],
        P[6],
        P[7],
        P[8],
        P[9],
        P[10],
        P[12],
        P[15]
    ])
    FY_calculated = pacejka(P_noCamber, P_camber, L, FZ, IA, alpha)
    FY_diff_total = 0
    for i in range(len(FY_measured)):
        FY_diff_total += abs(FY_calculated[i] - FY_measured[i])
    return FY_diff_total / len(FY_measured)
    
    
#want to minimize the difference between the measured and calculated cornering force values
def diffPacejka(P):
    FY_calculated = pacejka(P, FZ, IA, alpha)
    #must return a scalar, so return the average difference between calculated and measured
    FY_diff_total = 0
    for i in range(len(FY_measured)):
        FY_diff_total += abs(FY_calculated[i] - FY_measured[i])
    return FY_diff_total / len(FY_measured)

def remove_small_data(P):
    for i in range(len(P)):
        if abs(P[i]) < 1e-10:
            P[i] = 0

In [59]:
# Define the search space - range of values for each parameter
P = [250, 1.4, 2.4, -0.25, 3, -0.1, -1.5, 0, 0, -30.5, 1.15, 1, 0, 0, -0.128, 0, 0, 0, 1.43]
space = [Real(250.0, 250.1, name='p0'), 
         Real(-20.0, 20.0, name='p1'), 
         Real(-20.0, 20.0, name='p2'), 
         Real(-20.0, 20.0, name='p3'), 
         Real(-20.0, 20.0, name='p4'), 
         Real(-20.0, 20.0, name='p5'), 
         Real(-20.0, 20.0, name='p6'), 
         Real(-1e-10, 1e-10, name='p7'), 
         Real(-1e-10, 1e-10, name='p8'), 
         Real(-100.0, 100.0, name='p9'), 
         Real(-20.0, 20.0, name='p10'), 
         Real(-20.0, 20.0, name='p11'), 
         Real(-1e-10, 1e-10, name='p12'), 
         Real(-1e-10, 1e-10, name='p13'), 
         Real(-20.0, 20.0, name='p14'), 
         Real(-1e-10, 1e-10, name='p15'), 
         Real(-1e-10, 1e-10, name='p16'), 
         Real(-1e-10, 1e-10, name='p17'), 
         Real(-20.0, 20.0, name='p18')]

df = pd.read_csv("C:/Users/ajsau/Downloads/R20_FZ_250_filtered.csv")
FZ = df["NormalForce"]
IA = df["InclinationAngle"]
alpha = df["SlipAngle"]
FY_measured = df["LateralForce"]
L = np.array([1, 1, 1, 1, 1, 1, 1, 1])
result = gp_minimize(pacejka,      # The function to minimize
                     space,                   # The search space
                     n_calls=50,              # The number of evaluations
                     random_state=42)
print(f"Parameters recieved: {result.x}")
print(f"Parameters from brennan: {P}")
'''
# Perform Bayesian Optimization
result_noCamber_init = gp_minimize(diffPacejka_noCamber,      # The function to minimize
                     space_noCamber,                   # The search space
                     n_calls=50,              # The number of evaluations
                     random_state=42)         # Random state for reproducibility
P_noCamber_tmp = result_noCamber_init.x
P = edit_P_with_no_Camber(P, P_noCamber_tmp)
remove_small_data(P)
print(f"Best parameters from init optimization: {P}")

result_camber = gp_minimize(diffPacejka_camber,      # The function to minimize
                     space_camber,                   # The search space
                     n_calls=50,              # The number of evaluations
                     random_state=42)   
P_camber_tmp = result_camber.x
P = edit_P_with_camber(P, P_camber_tmp)
remove_small_data(P)
print(f"Best parameters from camber optimization: {P}")

result_noCamber_final = gp_minimize(diffPacejka_noCamber,      # The function to minimize
                     space_noCamber,                   # The search space
                     n_calls=50,              # The number of evaluations
                     random_state=42)         # Random state for reproducibility
P_noCamber_tmp = result_noCamber_final.x
P = edit_P_with_no_Camber(P, P_noCamber_tmp)
remove_small_data(P)
print(f"Best parameters from optimization: {P}")
print("Minimum value: {:.4f}".format(result.fun))
'''
# Plot convergence
plot_convergence(result)

ValueError: `func` should return a scalar